In [1]:
import os
import json 
from rouge_score import rouge_scorer
from backend.app.modules.document_preprocessing import PDFExtraction, CleanText, split_text
from backend.app.modules.document_retrieval import store_embeddings, retrieve_text
from backend.app.modules.chatbot_logic import Chatbot 

In [ ]:
import os 
import re
import json
import pandas as pd 
import numpy as np
from typing import List, Dict, Any, Optional
from langchain_community.document_loaders import PyPDFLoader
from backend.app.config import (
    OPENAI_KEY,
    MINI_LM_EMBED,
    OPENAI_EMBED,
    VECTOR_DB_PATH,
    PROJECT_ROOT
)
DEFAULT_EMBED_MODEL = OPENAI_EMBED

In [4]:
question_answer_pairs = [
  {
    "question": "Which further imaging is indicated for persistent hip pain over 6 weeks despite normal X-ray findings?",
    "answer_reference": "Recommendation: MRI for persistent symptoms with normal X-ray findings."
  },
  {
    "question": "When should an MRI of both hip joints be performed?",
    "answer_reference": "Recommendation: MRI of both hips for unilateral femoral head necrosis in ARCO stages I-IV."
  },
  {
    "question": "Which classification is recommended for staging atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: Use of the modified ARCO classification."
  },
  {
    "question": "What should be done if a subchondral fracture in ARCO Stage II is suspected, but the diagnosis is unclear?",
    "answer_reference": "Recommendation: Perform a CT scan to clarify the subchondral fracture."
  },
  {
    "question": "Should scintigraphy be used to diagnose atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: Scintigraphy is not recommended for the diagnosis of atraumatic femoral head necrosis."
  },
  {
    "question": "How does one differentiate between transient bone marrow edema and osteonecrosis on MRI?",
    "answer_reference": "Recommendation: MRI patterns and clinical course are crucial for differentiation."
  },
  {
    "question": "Which imaging method is considered the gold standard for diagnosing atraumatic femoral head necrosis?",
    "answer_reference": "Recommendation: MRI as the gold standard."
  },
  {
    "question": "Which imaging technique is best suited for detecting a subchondral fracture?",
    "answer_reference": "Recommendation: CT for visualizing subchondral fractures."
  },
  {
    "question": "What risk factors indicate bilateral involvement in femoral head necrosis?",
    "answer_reference": "Recommendation: Unilateral femoral head necrosis increases the risk of bilateral disease; consider risk factors."
  },
  {
    "question": "What radiological findings characterize ARCO Stage III?",
    "answer_reference": "Recommendation: Signs of a subchondral fracture with incipient articular surface incongruity on the radiograph."
  }
]

qa_dictionary = {}

for item in question_answer_pairs:
    question = item["question"]
    answer = item["answer_reference"]
    qa_dictionary[question] = answer

print(qa_dictionary)

{'Which further imaging is indicated for persistent hip pain over 6 weeks despite normal X-ray findings?': 'Recommendation: MRI for persistent symptoms with normal X-ray findings.', 'When should an MRI of both hip joints be performed?': 'Recommendation: MRI of both hips for unilateral femoral head necrosis in ARCO stages I-IV.', 'Which classification is recommended for staging atraumatic femoral head necrosis?': 'Recommendation: Use of the modified ARCO classification.', 'What should be done if a subchondral fracture in ARCO Stage II is suspected, but the diagnosis is unclear?': 'Recommendation: Perform a CT scan to clarify the subchondral fracture.', 'Should scintigraphy be used to diagnose atraumatic femoral head necrosis?': 'Recommendation: Scintigraphy is not recommended for the diagnosis of atraumatic femoral head necrosis.', 'How does one differentiate between transient bone marrow edema and osteonecrosis on MRI?': 'Recommendation: MRI patterns and clinical course are crucial for

### Document Extraction & Splitting

In [5]:
def extract_text_from_pdf(file_path: str) -> str:
        """
        Extract text from a PDF file using PyPDFLoader.
        """
        try:
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            return [page.page_content for page in documents]
        except Exception as e: 
            raise Exception(f"Error extracting text from PDF: {e}")
        
class CleanText:
    """Simplified text cleaner for medical documents."""
    
    def __init__(self, text: str):
        self.text = text
        
        # Common patterns to remove
        self.header_patterns = [
            r'^S3-Leitlinie.*?Langfassung\s+Version vom \d{2}\.\d{2}\.\d{4}.*?\n',
            r'^Seite \d+ von \d+'
        ]
        self.footer_patterns = [
            r'\n\d+\s*$',  # Page numbers at end
            r'©.*?$'       # Copyright notices
        ]
        
    def remove_unwanted_patterns(self) -> 'CleanText':
        """Remove headers, footers, and other unwanted patterns."""
        all_patterns = self.header_patterns + self.footer_patterns
        for pattern in all_patterns:
            self.text = re.sub(pattern, '', self.text, flags=re.MULTILINE)
        return self
    
    def clean_special_chars(self) -> 'CleanText':
        """Remove unusual characters while keeping basic punctuation."""
        # Keep letters, numbers, basic punctuation, and whitespace
        self.text = re.sub(r'[^\w\s.,;:\-()/°]', '', self.text)
        return self
    
    def fix_spacing(self) -> 'CleanText':
        """Clean up spacing and line breaks."""
        # Replace multiple spaces with single space
        self.text = re.sub(r' +', ' ', self.text)
        # Replace single newlines with space (keep paragraphs)
        self.text = re.sub(r'(?<!\n)\n(?!\n)', ' ', self.text)
        # Remove space around hyphens
        self.text = re.sub(r'\s*-\s*', '-', self.text)
        return self
    
    def clean(self) -> str:
        """Run the complete cleaning process."""
        return (
            self.remove_unwanted_patterns()
            .clean_special_chars()
            .fix_spacing()
            .text
        )



In [ ]:
pdf_path = os.path.join(os.getcwd(), "backend", "app", "documents", "Guideline_atraumatische_Femurkopfnekrose_2019-09_1-abgelaufen.pdf")

pdf_extraction = extract_text_from_pdf(pdf_path)
print(pdf_extraction)


